In [10]:
from pathlib import Path

QUESTION_BANK_DIR = Path(
    r"C:\Users\User\RAG SYS\Question_Generation\knowledge_base\questions"
)

print("Current directory:", Path.cwd())
print("Question Bank:", QUESTION_BANK_DIR)
print("Exists:", QUESTION_BANK_DIR.exists())

Current directory: C:\Users\User\RAG SYS\Question_Generation\ingestion
Question Bank: C:\Users\User\RAG SYS\Question_Generation\knowledge_base\questions
Exists: True


In [11]:
from pathlib import Path
import re
from typing import Any

QUESTION_BANK_DIR = Path.cwd().parent / "knowledge_base" / "questions"

print("Question Bank:", QUESTION_BANK_DIR.resolve())
print("Exists:", QUESTION_BANK_DIR.exists())

Question Bank: C:\Users\User\RAG SYS\Question_Generation\knowledge_base\questions
Exists: True


In [15]:
from pathlib import Path
import re
from typing import Any


# ============================================================
# CONFIGURATION
# ============================================================

# Current notebook working directory:
#
# RAG SYS/
# └── Question_Generation/
#     └── ingestion/    <-- current directory
#
# Therefore:
# Path.cwd().parent
#       ↓
# Question_Generation/
#
# Then:
# knowledge_base/questions/

QUESTION_BANK_DIR = (
    Path.cwd().parent
    / "knowledge_base"
    / "questions"
)


# ============================================================
# REQUIRED FIELDS
# ============================================================

REQUIRED_FIELDS = [
    "type",
    "track",
    "category",
    "topic",
    "difficulty",
    "experience",
    "skills",
    "language",
    "duration",
    "source",
]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_value(value: str) -> str:
    """Clean a metadata value."""
    return value.strip()


def parse_list(value: str) -> list[str]:
    """
    Convert comma-separated metadata into a Python list.

    Example:
        'intern, junior'
        ->
        ['intern', 'junior']
    """

    if not value:
        return []

    return [
        item.strip()
        for item in value.split(",")
        if item.strip()
    ]


def parse_duration(value: str) -> int | None:
    """
    Convert duration to an integer.

    Examples:
        '60'          -> 60
        '60 sec'      -> 60
        '90 seconds'  -> 90
    """

    if not value:
        return None

    match = re.search(r"\d+", value)

    if match:
        return int(match.group())

    return None


def normalize_id_part(value: str) -> str:
    """
    Convert a string into a safe ID component.

    Example:
        'Machine Learning'
        ->
        'machine_learning'
    """

    value = value.strip().lower()

    value = re.sub(
        r"[^a-z0-9]+",
        "_",
        value
    )

    value = value.strip("_")

    return value


# ============================================================
# EXTRACT METADATA
# ============================================================

def extract_metadata(section: str) -> dict[str, str]:
    """
    Extract metadata from a question section.

    Expected format:

    - Type: technical
    - Track: ai_ml
    - Category: machine_learning
    """

    metadata: dict[str, str] = {}

    pattern = re.compile(
        r"^-\s*([A-Za-z_ ]+?)\s*:\s*(.*?)\s*$",
        re.MULTILINE
    )

    for match in pattern.finditer(section):

        field = match.group(1).strip()

        value = match.group(2).strip()

        field = field.lower()

        field = field.replace(" ", "_")

        metadata[field] = value

    return metadata


# ============================================================
# EXTRACT QUESTION TEXT
# ============================================================

def extract_question_text(section: str) -> str:
    """
    Extract content under:

    ### Question
    """

    pattern = re.compile(
        r"###\s*Question\s*\n+(.*?)(?=\n###|\Z)",
        re.DOTALL | re.IGNORECASE
    )

    match = pattern.search(section)

    if not match:
        return ""

    return match.group(1).strip()


# ============================================================
# EXTRACT EXPECTED CONCEPTS
# ============================================================

def extract_expected_concepts(
    section: str
) -> list[str]:

    """
    Extract bullet points under:

    ### Expected Concepts
    """

    pattern = re.compile(
        r"###\s*Expected Concepts\s*\n+(.*?)(?=\n###|\Z)",
        re.DOTALL | re.IGNORECASE
    )

    match = pattern.search(section)

    if not match:
        return []

    concepts_text = match.group(1)

    concepts = []

    for line in concepts_text.splitlines():

        line = line.strip()

        if line.startswith("-"):

            concept = line[1:].strip()

            if concept:
                concepts.append(concept)

    return concepts


# ============================================================
# PARSE ONE QUESTION
# ============================================================

def parse_question_section(
    section: str,
    file_path: Path
) -> dict[str, Any] | None:

    """
    Parse one complete question.
    """

    # --------------------------------------------------------
    # QUESTION HEADING
    # --------------------------------------------------------
    #
    # Example:
    #
    # ## Q001 — What is Computer Vision?
    #

    heading_pattern = re.compile(
        r"^##\s+(Q\d+)\s*[—-]\s*(.+?)\s*$",
        re.MULTILINE
    )

    heading_match = heading_pattern.search(section)

    if not heading_match:
        return None

    original_id = heading_match.group(1).strip()

    title = heading_match.group(2).strip()


    # --------------------------------------------------------
    # METADATA
    # --------------------------------------------------------

    metadata = extract_metadata(section)


    # --------------------------------------------------------
    # QUESTION
    # --------------------------------------------------------

    question_text = extract_question_text(section)


    # --------------------------------------------------------
    # EXPECTED CONCEPTS
    # --------------------------------------------------------

    expected_concepts = extract_expected_concepts(
        section
    )


    # --------------------------------------------------------
    # FILE INFORMATION
    # --------------------------------------------------------

    try:

        relative_path = file_path.relative_to(
            QUESTION_BANK_DIR
        )

    except ValueError:

        relative_path = file_path


    parts = relative_path.parts


    # --------------------------------------------------------
    # TRACK
    # --------------------------------------------------------

    track = metadata.get(
        "track",
        ""
    ).strip()


    # --------------------------------------------------------
    # CATEGORY
    # --------------------------------------------------------

    category = metadata.get(
        "category",
        ""
    ).strip()


    # --------------------------------------------------------
    # FALLBACK TRACK
    # --------------------------------------------------------

    if not track and len(parts) >= 2:

        track = parts[0]


    # --------------------------------------------------------
    # FALLBACK CATEGORY
    # --------------------------------------------------------

    if not category and len(parts) >= 2:

        category = Path(
            parts[-1]
        ).stem


    # --------------------------------------------------------
    # FILE NAME
    # --------------------------------------------------------

    file_stem = normalize_id_part(
        Path(parts[-1]).stem
    )


    # --------------------------------------------------------
    # CREATE GLOBAL UNIQUE ID
    # --------------------------------------------------------
    #
    # IMPORTANT:
    #
    # We include the actual Markdown filename.
    #
    # Example:
    #
    # devops/ci_cd.md
    # Q007
    #
    # becomes:
    #
    # devops_ci_cd_q007
    #
    # While:
    #
    # devops/devops_basics.md
    # Q007
    #
    # becomes:
    #
    # devops_devops_basics_q007
    #
    # Therefore they cannot collide.

    unique_id = "_".join(
        [
            normalize_id_part(track),
            file_stem,
            normalize_id_part(original_id)
        ]
    )


    # --------------------------------------------------------
    # BUILD STRUCTURED QUESTION
    # --------------------------------------------------------

    question = {

        # Globally unique system ID
        "id": unique_id,

        # Original Markdown ID
        "original_id": original_id,

        # Heading title
        "title": title,

        # Actual interview question
        "question": question_text,

        # Metadata
        "type": metadata.get(
            "type",
            ""
        ),

        "track": track,

        "category": category,

        "topic": metadata.get(
            "topic",
            ""
        ),

        "difficulty": metadata.get(
            "difficulty",
            ""
        ),

        # Lists
        "experience": parse_list(
            metadata.get(
                "experience",
                ""
            )
        ),

        "skills": parse_list(
            metadata.get(
                "skills",
                ""
            )
        ),

        # Language
        "language": metadata.get(
            "language",
            "en"
        ),

        # Duration
        "duration": parse_duration(
            metadata.get(
                "duration",
                ""
            )
        ),

        # Source
        "source": metadata.get(
            "source",
            "question_bank"
        ),

        # Expected concepts
        "expected_concepts": expected_concepts,

        # Original Markdown file
        "source_file": str(
            relative_path
        ),
    }


    return question


# ============================================================
# PARSE ONE MARKDOWN FILE
# ============================================================

def parse_markdown_file(
    file_path: Path
) -> list[dict[str, Any]]:

    """
    Parse all questions inside one Markdown file.
    """

    content = file_path.read_text(
        encoding="utf-8"
    )


    # --------------------------------------------------------
    # Split Markdown into question sections
    # --------------------------------------------------------

    sections = re.split(
        r"(?=^##\s+Q\d+\s*[—-])",
        content,
        flags=re.MULTILINE
    )


    questions = []


    for section in sections:

        section = section.strip()

        if not section:
            continue


        question = parse_question_section(
            section,
            file_path
        )


        if question:

            questions.append(
                question
            )


    return questions


# ============================================================
# PARSE ENTIRE QUESTION BANK
# ============================================================

def parse_question_bank(
    question_bank_dir: Path = QUESTION_BANK_DIR
) -> tuple[
    list[dict[str, Any]],
    list[dict[str, Any]]
]:

    """
    Parse every Markdown file in the Question Bank.

    Returns:

        questions
        errors
    """

    all_questions: list[dict[str, Any]] = []

    errors: list[dict[str, Any]] = []


    # --------------------------------------------------------
    # Check directory
    # --------------------------------------------------------

    if not question_bank_dir.exists():

        raise FileNotFoundError(
            f"Question Bank not found:\n"
            f"{question_bank_dir}"
        )


    # --------------------------------------------------------
    # Find Markdown files
    # --------------------------------------------------------

    markdown_files = sorted(
        question_bank_dir.rglob("*.md")
    )


    print(
        f"\nFound {len(markdown_files)} "
        f"Markdown files."
    )


    # --------------------------------------------------------
    # Parse each file
    # --------------------------------------------------------

    for file_path in markdown_files:

        # Relative path for checking hidden folders
        relative_file = file_path.relative_to(
            question_bank_dir
        )


        # ----------------------------------------------------
        # Ignore hidden directories
        # ----------------------------------------------------

        if any(
            part.startswith(".")
            for part in relative_file.parts
        ):

            continue


        try:

            questions = parse_markdown_file(
                file_path
            )


            all_questions.extend(
                questions
            )


            print(
                f"✓ {relative_file}"
                f" → {len(questions)} questions"
            )


        except Exception as error:

            error_info = {

                "file": str(
                    relative_file
                ),

                "error": str(error)
            }


            errors.append(
                error_info
            )


            print(
                f"✗ {relative_file}"
                f" → {error}"
            )


    return (
        all_questions,
        errors
    )


# ============================================================
# VALIDATE QUESTIONS
# ============================================================

def validate_questions(
    questions: list[dict[str, Any]]
) -> tuple[
    list[dict[str, Any]],
    dict[str, list[str]],
    dict[str, list[str]]
]:

    """
    Validate parsed questions.

    Returns:

        valid_questions
        missing_fields
        duplicate_ids
    """

    valid_questions = []

    missing_fields: dict[str, list[str]] = {}

    id_locations: dict[str, list[str]] = {}


    # --------------------------------------------------------
    # Validate each question
    # --------------------------------------------------------

    for question in questions:

        missing = []


        # ----------------------------------------------------
        # Required metadata
        # ----------------------------------------------------

        for field in REQUIRED_FIELDS:

            value = question.get(
                field
            )


            if (
                value is None
                or value == ""
                or value == []
            ):

                missing.append(
                    field
                )


        # ----------------------------------------------------
        # Question text
        # ----------------------------------------------------

        if not question.get(
            "question"
        ):

            missing.append(
                "question"
            )


        # ----------------------------------------------------
        # Store missing fields
        # ----------------------------------------------------

        if missing:

            missing_fields[
                question["id"]
            ] = missing

        else:

            valid_questions.append(
                question
            )


        # ----------------------------------------------------
        # Track IDs
        # ----------------------------------------------------

        id_locations.setdefault(
            question["id"],
            []
        ).append(
            question["source_file"]
        )


    # --------------------------------------------------------
    # Find duplicate IDs
    # --------------------------------------------------------

    duplicate_ids = {

        question_id: files

        for question_id, files
        in id_locations.items()

        if len(files) > 1
    }


    return (
        valid_questions,
        missing_fields,
        duplicate_ids
    )


# ============================================================
# PRINT SAMPLE QUESTION
# ============================================================

def print_sample_question(
    question: dict[str, Any]
) -> None:

    print(
        "\n" + "=" * 70
    )

    print(
        "SAMPLE PARSED QUESTION"
    )

    print(
        "=" * 70
    )


    for key, value in question.items():

        print(
            f"\n{key}:"
        )

        print(
            f"  {value}"
        )


# ============================================================
# MAIN
# ============================================================

def main():

    print(
        "=" * 70
    )

    print(
        "QUESTION BANK PARSER"
    )

    print(
        "=" * 70
    )


    # --------------------------------------------------------
    # Show current directory
    # --------------------------------------------------------

    print(
        "\nCurrent directory:"
    )

    print(
        Path.cwd()
    )


    # --------------------------------------------------------
    # Show Question Bank
    # --------------------------------------------------------

    print(
        "\nQuestion Bank:"
    )

    print(
        QUESTION_BANK_DIR.resolve()
    )


    # --------------------------------------------------------
    # Check Question Bank
    # --------------------------------------------------------

    print(
        "\nQuestion Bank exists:"
    )

    print(
        QUESTION_BANK_DIR.exists()
    )


    if not QUESTION_BANK_DIR.exists():

        print(
            "\n❌ Question Bank directory "
            "does not exist."
        )

        return


    # --------------------------------------------------------
    # Parse Question Bank
    # --------------------------------------------------------

    questions, errors = parse_question_bank()


    # --------------------------------------------------------
    # Validate
    # --------------------------------------------------------

    (
        valid_questions,
        missing_fields,
        duplicate_ids
    ) = validate_questions(
        questions
    )


    # ========================================================
    # SUMMARY
    # ========================================================

    print(
        "\n" + "=" * 70
    )

    print(
        "PARSING SUMMARY"
    )

    print(
        "=" * 70
    )


    # Count only non-hidden Markdown files
    visible_markdown_files = [
        file
        for file in QUESTION_BANK_DIR.rglob("*.md")
        if not any(
            part.startswith(".")
            for part in file.relative_to(
                QUESTION_BANK_DIR
            ).parts
        )
    ]


    print(
        f"\nMarkdown files:"
        f"        {len(visible_markdown_files)}"
    )

    print(
        f"Questions parsed:"
        f"       {len(questions)}"
    )

    print(
        f"Valid questions:"
        f"        {len(valid_questions)}"
    )

    print(
        f"Questions with issues:"
        f"  {len(questions) - len(valid_questions)}"
    )

    print(
        f"File parsing errors:"
        f"    {len(errors)}"
    )

    print(
        f"Duplicate IDs:"
        f"           {len(duplicate_ids)}"
    )


    # ========================================================
    # MISSING FIELDS
    # ========================================================

    print(
        "\n" + "=" * 70
    )

    print(
        "MISSING FIELDS"
    )

    print(
        "=" * 70
    )


    if missing_fields:

        for question_id, fields in missing_fields.items():

            print(
                f"\n⚠ {question_id}"
            )

            print(
                f"   Missing: {', '.join(fields)}"
            )

    else:

        print(
            "None 🎉"
        )


    # ========================================================
    # DUPLICATE IDs
    # ========================================================

    print(
        "\n" + "=" * 70
    )

    print(
        "DUPLICATE IDs"
    )

    print(
        "=" * 70
    )


    if duplicate_ids:

        for question_id, files in duplicate_ids.items():

            print(
                f"\n⚠ {question_id}"
            )

            for file in files:

                print(
                    f"   - {file}"
                )

    else:

        print(
            "None 🎉"
        )


    # ========================================================
    # FILE ERRORS
    # ========================================================

    print(
        "\n" + "=" * 70
    )

    print(
        "FILE ERRORS"
    )

    print(
        "=" * 70
    )


    if errors:

        for error in errors:

            print(
                f"\n⚠ {error['file']}"
            )

            print(
                f"   {error['error']}"
            )

    else:

        print(
            "None 🎉"
        )


    # ========================================================
    # SAMPLE QUESTION
    # ========================================================

    if questions:

        print_sample_question(
            questions[0]
        )


    # ========================================================
    # COMPLETE
    # ========================================================

    print(
        "\n" + "=" * 70
    )

    print(
        "PARSER COMPLETE"
    )

    print(
        "=" * 70
    )


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    main()

QUESTION BANK PARSER

Current directory:
C:\Users\User\RAG SYS\Question_Generation\ingestion

Question Bank:
C:\Users\User\RAG SYS\Question_Generation\knowledge_base\questions

Question Bank exists:
True

Found 40 Markdown files.
✓ ai_ml\computer_vision.md → 50 questions
✓ ai_ml\deep_learning.md → 100 questions
✓ ai_ml\llm_rag.md → 50 questions
✓ ai_ml\machine_learning.md → 100 questions
✓ ai_ml\nlp.md → 50 questions
✓ backend\databases.md → 50 questions
✓ backend\django.md → 50 questions
✓ backend\fastapi.md → 50 questions
✓ backend\python.md → 50 questions
✓ backend\rest_api.md → 50 questions
✓ behavioral\conflict.md → 50 questions
✓ behavioral\leadership.md → 50 questions
✓ behavioral\problem_solving.md → 50 questions
✓ behavioral\teamwork.md → 50 questions
✓ cloud\cloud_basics.md → 50 questions
✓ cs_fundamentals\algorithms.md → 50 questions
✓ cs_fundamentals\data_structures.md → 50 questions
✓ cs_fundamentals\databases.md → 50 questions
✓ cs_fundamentals\networking.md → 50 question

In [16]:
# ============================================================
# GET PARSED QUESTIONS FOR EMBEDDING
# ============================================================

questions, errors = parse_question_bank()

valid_questions, missing_fields, duplicate_ids = validate_questions(
    questions
)

print("=" * 70)
print("PARSER OUTPUT FOR EMBEDDING")
print("=" * 70)

print(f"\nTotal parsed questions: {len(questions)}")
print(f"Valid questions:        {len(valid_questions)}")
print(f"Errors:                 {len(errors)}")
print(f"Missing fields:         {len(missing_fields)}")
print(f"Duplicate IDs:          {len(duplicate_ids)}")


Found 40 Markdown files.
✓ ai_ml\computer_vision.md → 50 questions
✓ ai_ml\deep_learning.md → 100 questions
✓ ai_ml\llm_rag.md → 50 questions
✓ ai_ml\machine_learning.md → 100 questions
✓ ai_ml\nlp.md → 50 questions
✓ backend\databases.md → 50 questions
✓ backend\django.md → 50 questions
✓ backend\fastapi.md → 50 questions
✓ backend\python.md → 50 questions
✓ backend\rest_api.md → 50 questions
✓ behavioral\conflict.md → 50 questions
✓ behavioral\leadership.md → 50 questions
✓ behavioral\problem_solving.md → 50 questions
✓ behavioral\teamwork.md → 50 questions
✓ cloud\cloud_basics.md → 50 questions
✓ cs_fundamentals\algorithms.md → 50 questions
✓ cs_fundamentals\data_structures.md → 50 questions
✓ cs_fundamentals\databases.md → 50 questions
✓ cs_fundamentals\networking.md → 50 questions
✓ cs_fundamentals\oop.md → 50 questions
✓ cs_fundamentals\operating_systems.md → 50 questions
✓ cybersecurity\Cybersecurity.md → 100 questions
✓ data_science\data_analysis.md → 50 questions
✓ data_scien

In [13]:
import pickle
from pathlib import Path

# Question_Generation/
PROJECT_ROOT = Path.cwd().parent

# Create data folder
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# File where we will save the parsed questions
OUTPUT_FILE = DATA_DIR / "parsed_questions.pkl"

# Save the 2,150 validated questions
with open(OUTPUT_FILE, "wb") as f:
    pickle.dump(valid_questions, f)

print("=" * 70)
print("PARSED QUESTIONS SAVED")
print("=" * 70)

print(f"\nFile:")
print(OUTPUT_FILE)

print(f"\nQuestions saved:")
print(len(valid_questions))

print("\n✓ Save complete")

PARSED QUESTIONS SAVED

File:
C:\Users\User\RAG SYS\Question_Generation\data\parsed_questions.pkl

Questions saved:
2150

✓ Save complete
